# 영화 리뷰 워드 임베딩 (Word2Vec, FastText)
- gensim 라이브러리 사용 : pip install gensim
    - Word2Vec : models.Word2Vec
    - FastText : models.FastText

## 1. 데이터 준비
* 토큰화가 잘 되어 있는 filtered 데이터 사용

In [3]:
import pandas as pd
data_filename = './data/Korean_movie_reviews_2016_filtered.csv'
data_df = pd.read_csv(data_filename)
data_df.head()

,review,rate
0,아니 딴 그렇 비 비탄 총 대체 왜 들 온겨,7
1,진심 쓰레기 영화 만들 무서 알 쫄아 틀었 이건 뭐 웃 거리 없는 쓰레기 영화 임,1
2,역대 좀비 영화 가장 최고다 원작 만화 읽어 보려 영화 보고 결정 하려 감독 간츠 ...,10
3,온종일 불편한 피 범벅 일,6
4,답답함 극치 움직일 잇으 좀 움직여 어지간히 좀비 봣으 얼 타고 때려 잡 때 되 않냐,1


In [7]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 788189 entries, 0 to 788188
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  785448 non-null  object
 1   rate    788189 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 12.0+ MB


In [13]:
# 결측치 제거
data_df.dropna(inplace=True)

In [16]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 785448 entries, 0 to 788188
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  785448 non-null  object
 1   rate    785448 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 18.0+ MB


In [17]:
# review만 모아서 review별 토큰 리스트로 변환 
review_list = list(data_df.review)
review_list[:5]

['아니 딴 그렇 비 비탄 총 대체 왜 들 온겨',
 '진심 쓰레기 영화 만들 무서 알 쫄아 틀었 이건 뭐 웃 거리 없는 쓰레기 영화 임',
 '역대 좀비 영화 가장 최고다 원작 만화 읽어 보려 영화 보고 결정 하려 감독 간츠 실사 했 사람 거르려 그냥 봤 정말 흠잡 없는 최고 좀비 영화 잔인 거 싫어하지 참고 볼 만하 로미 인물 왜 그런 모르',
 '온종일 불편한 피 범벅 일',
 '답답함 극치 움직일 잇으 좀 움직여 어지간히 좀비 봣으 얼 타고 때려 잡 때 되 않냐']

In [18]:
type(review_list[0])

str

In [21]:
# 결측치 제거 (rate에 결측치)
# 결측치를 제거하지 않으면 review 데이터 추출 시 float으로 타입 변환됨
token_list = []
for review in review_list:
    if review:
        token_list.append(review.split())
print(token_list[:2])

[['아니', '딴', '그렇', '비', '비탄', '총', '대체', '왜', '들', '온겨'], ['진심', '쓰레기', '영화', '만들', '무서', '알', '쫄아', '틀었', '이건', '뭐', '웃', '거리', '없는', '쓰레기', '영화', '임']]


## 1. Word2Vec 활용 영화 리뷰 워드 임베딩
* https://radimrehurek.com/gensim/models/word2vec.html

### Skipgram, negative=10 인 경우

In [22]:
# Word2Vec 모델 생성 및 학습 : window=3, min_count=3
from gensim.models import Word2Vec
model_sg_n10 = Word2Vec(token_list, window=3, min_count=3, sg=1, negative=10, vector_size=100)

In [42]:
# 단어의 임베딩 벡터 확인
model_sg_n10.wv['이정재']

array([-0.29766372,  0.2585942 , -0.24562654, -0.3432096 , -0.16244367,
       -0.27551797, -0.03771994,  0.23487943,  0.9566788 , -0.03606756,
        0.10007322, -0.4025389 , -0.3121363 ,  0.4323223 , -0.09891988,
       -0.034065  , -0.09032638,  0.08281866,  0.49256784, -0.46024793,
        0.14401193,  0.1707471 ,  0.04813226,  0.27365816, -0.369089  ,
       -0.31392154,  0.03351332, -0.14030616, -0.11182987, -0.12733558,
        0.0781903 ,  0.05541217,  0.1354447 , -0.26731908,  0.04472621,
       -0.3712244 , -0.3137214 , -0.16196567, -0.56607884, -0.3881474 ,
       -0.48591566,  0.13041508, -0.1218045 , -0.27313587,  0.03154811,
       -0.11534435, -0.04341701, -0.35952562, -0.5500355 , -0.13420656,
        0.1964621 , -0.8156866 , -0.01402174, -0.1850836 ,  0.19116603,
       -0.11301617,  0.3504573 ,  0.21988398,  0.24563302, -0.03863401,
       -0.22627501,  0.10201882,  0.39509454, -0.15666391, -0.6499282 ,
       -0.24430138, -0.18280844, -0.694166  ,  0.3770538 , -0.33

In [24]:
# 단어의 임베딩 벡터 차원 확인
len(model_sg_n10.wv['이정재'])

100

In [25]:
# 두 단어 간 유사도 확인
wv = model_sg_n10.wv
wv.similarity('이정재', '정우성')

np.float32(0.77049935)

In [27]:
# 특정 단어와 유사한 단어 추출
wv.most_similar('이정재', topn=20)

[('이범수', 0.806997537612915),
 ('송강호', 0.784257709980011),
 ('공유', 0.7751954793930054),
 ('정우성', 0.7704992890357971),
 ('이성민', 0.7645677924156189),
 ('박해일', 0.758590817451477),
 ('김범수', 0.7408427596092224),
 ('조재현', 0.7261012196540833),
 ('이병헌', 0.7254420518875122),
 ('리암', 0.718906044960022),
 ('오지호', 0.7183159589767456),
 ('주지훈', 0.7143629193305969),
 ('곽도원', 0.7083498239517212),
 ('이진욱', 0.7043948769569397),
 ('박철민', 0.6971664428710938),
 ('요한', 0.6968011856079102),
 ('마동석', 0.6957160234451294),
 ('김명민', 0.6925097107887268),
 ('하정우', 0.6914356350898743),
 ('류승범', 0.6913017630577087)]

In [28]:
wv.most_similar('재밌', topn=20)

[('재미있', 0.9071133732795715),
 ('재밌었', 0.8345319032669067),
 ('재밌네', 0.8238674998283386),
 ('재밌어', 0.808509886264801),
 ('재밋음', 0.8082528710365295),
 ('잼남', 0.8037970662117004),
 ('잼슴', 0.7949492931365967),
 ('재밋엇음', 0.7856809496879578),
 ('재밌아', 0.7840187549591064),
 ('재밋었음', 0.7823809385299683),
 ('쟈밋', 0.776269257068634),
 ('재밋엇어용', 0.7727100253105164),
 ('재미있었', 0.7707850933074951),
 ('재밋어용', 0.7664986848831177),
 ('재밌슴', 0.765891969203949),
 ('재밋엇어', 0.7640035152435303),
 ('재밋구', 0.7605550289154053),
 ('재밋었어', 0.7583940625190735),
 ('재밋습니', 0.7555723190307617),
 ('재밋당', 0.7544749975204468)]

In [29]:
print([word for word, _ in wv.most_similar('재밌', topn=50)])

['재미있', '재밌었', '재밌네', '재밌어', '재밋음', '잼남', '잼슴', '재밋엇음', '재밌아', '재밋었음', '쟈밋', '재밋엇어용', '재미있었', '재밋어용', '재밌슴', '재밋엇어', '재밋구', '재밋었어', '재밋습니', '재밋당', '재밋엇', '재밋습', '재밋었습니', '존잼임', '재밋네용', '재밋게봣', '엇', '재밋네', '재미있더', '재밋어', '재밋게봣어', '재밋게봣습니', '재밋는듯', '재미있네', '꿀잼임', '잼난', '재밌드', '존잼', '재밋습니당', '재밋게봄', '짱재밋어', '재밌았', '재밋엇습니', '재미있겠', '닺', '재밌엇습니', '꿀잼', '재밌고', '재밌더', '재밋었']


### Skipgram, negative=5 인 경우

In [32]:
# 모델 생성
model_sg_n5 = Word2Vec(token_list, vector_size=100, window=3, min_count=3, sg=1, negative=5)

In [34]:
# 특어 단어와 유사한 단어 추출 : 이정재
wv = model_sg_n5.wv
wv.most_similar('이정재', topn=20)

[('이범수', 0.8345927000045776),
 ('송강호', 0.7931848168373108),
 ('공유', 0.7837538719177246),
 ('김범수', 0.7411858439445496),
 ('리암', 0.7260441780090332),
 ('김남길', 0.7174344062805176),
 ('정우성', 0.7164984941482544),
 ('이병헌', 0.7158967852592468),
 ('이성민', 0.7097455263137817),
 ('곽도원', 0.7073048949241638),
 ('박해일', 0.7051002979278564),
 ('김윤석', 0.6966869235038757),
 ('정재형', 0.6908308267593384),
 ('슨', 0.689393937587738),
 ('조재현', 0.6870311498641968),
 ('주지훈', 0.6855501532554626),
 ('마동석', 0.6839798092842102),
 ('김명민', 0.6824650168418884),
 ('작대기', 0.6719187498092651),
 ('조진웅', 0.6686755418777466)]

In [35]:
# 특어 단어와 유사한 단어 추출 : 재밌
print([word for word, _ in wv.most_similar('재밌', topn=50)])

['재미있', '재밋음', '재밌네', '재밌었', '잼남', '재밌어', '재밋습니', '재밋엇음', '재밋었음', '재밋엇', '재미있었', '재밌아', '재밋어용', '재밋었습니', '재밋네', '재밋엇어용', '잼슴', '재밋었어', '재밋었', '존잼임', '재밌슴', '쟈밋', '재밋구', '재미있더', '재밋엇어', '더재밋', '재미있네', '재밋게봣습니', '잼난', '재밋게봣어', '재밋게봣', '꿀잼임', '재밋어', '재미있겠', '재밌드', '재밋는듯', '재밋엇습니', '재밌고', '재밋습', '재밌더', '짱잼', '재밋습니당', '재밋당', '재밋게봄', '엇', '재미나다', '재미있구', '재밋네용', '재미있어', '번보']


### CBOW, negative=10 인 경우

In [36]:
model_cbow_n10 = Word2Vec(token_list, vector_size=100, window=3, min_count=3, sg=0, negative=10)

In [37]:
wv = model_cbow_n10.wv
wv.most_similar('이정재', topn=20)

[('이범수', 0.7806339263916016),
 ('김윤석', 0.7455747723579407),
 ('조재현', 0.7454347610473633),
 ('공유', 0.7396817803382874),
 ('김남길', 0.7194387912750244),
 ('이성민', 0.7192279100418091),
 ('주지훈', 0.7100092768669128),
 ('송강호', 0.7043404579162598),
 ('박해일', 0.7017472386360168),
 ('김범수', 0.6997104287147522),
 ('이진욱', 0.694035530090332),
 ('요한', 0.668479859828949),
 ('정우성', 0.65849769115448),
 ('엄지원', 0.6502476334571838),
 ('곽도원', 0.6500393748283386),
 ('송광호', 0.649560272693634),
 ('임시완', 0.6443723440170288),
 ('남자배우', 0.6416038870811462),
 ('류승범', 0.6407783627510071),
 ('하정우', 0.6397674083709717)]

In [38]:
print([word for word, _ in wv.most_similar('재밌', topn=50)])

['재미있', '재밌네', '재밌었', '재밌어', '재밋음', '재밌는', '재미있었', '잼남', '재미있네', '재밋어', '재밋네', '재밌더', '재밋엇어', '재미있어', '재밌던', '재밋', '재미있던', '꿀잼', '재밋엇', '재밌다', '재밋었', '재밌고', '재밋엇음', '재미있는', '재밋었어', '재밌구', '재밌게', '재미있더', '잼', '재미있다', '재밋었음', '재밋습니', '웃김', '재미있고', '개꿀잼', '존잼', '재미있게', '재밌겠', '재밌으', '괜찮', '재미나', '무서웠', '재밋었습니', '재밋어용', '재미있겠', '멋있', '재미없', '웃겼', '재미있구', '무서']


### CBOW, negative=5 인 경우

In [39]:
model_cbow_n5 = Word2Vec(token_list, vector_size=100, window=3, min_count=3, sg=0, negative=5)

In [40]:
wv = model_cbow_n5.wv
wv.most_similar('이정재', topn=20)

[('이범수', 0.792234480381012),
 ('공유', 0.7420986294746399),
 ('김윤석', 0.7160813212394714),
 ('송강호', 0.7097400426864624),
 ('주지훈', 0.7074684500694275),
 ('이성민', 0.707250714302063),
 ('조재현', 0.700630784034729),
 ('이진욱', 0.6989403367042542),
 ('김남길', 0.6905928254127502),
 ('김범수', 0.6825940012931824),
 ('곽도원', 0.6781647205352783),
 ('박해일', 0.6544505953788757),
 ('하정우', 0.6523354053497314),
 ('정우성', 0.6507932543754578),
 ('이병헌', 0.6397594213485718),
 ('김성균', 0.6394334435462952),
 ('김성오', 0.6389340162277222),
 ('조진웅', 0.6365129351615906),
 ('엄지원', 0.6312909126281738),
 ('공효진', 0.6312551498413086)]

In [41]:
print([word for word, _ in wv.most_similar('재밌', topn=50)])

['재미있', '재밌네', '재밌어', '재밌었', '재밋음', '재밌는', '재밋어', '재미있었', '재미있네', '재밋엇', '재밋엇어', '재밌더', '재밋네', '잼남', '꿀잼', '재밌던', '재밌다', '재미있어', '재밋', '재미있는', '재밌고', '재밋었', '재밌게', '재밋엇음', '재미있던', '재밌구', '재밋었어', '재미있다', '웃김', '잼', '재미있더', '재밋습니', '재밋었음', '개꿀잼', '재밌으', '재미있고', '재미있게', '재밌겠', '재미없', '존잼', '웃겼', '괜찮', '무서웠', '재미나', '재미있겠', '졸잼', '멋있', '핵꿀잼', '재밋었습니', '재밋어용']


### OOV(Out of Vocabulary) 문제

In [44]:
# corpus에 없는 단어 확인 : 우주평화 
'우주평화' in model_sg_n10.wv.key_to_index

False

In [43]:
# corpus에 없는 단어의 임베딩 벡터 확인 
model_sg_n10.wv['우주평화']

KeyError: "Key '우주평화' not present"

## 2. FastText 활용 영화 리뷰 워드 임베딩
* https://radimrehurek.com/gensim/models/fasttext.html

In [45]:
# FastText 모델 생성 및 학습
# window=3, min_count=3, min_n=2, max_n=2
from gensim.models import FastText
model = FastText(token_list, vector_size=100, window=3, min_count=3, sg=1, negative=10, min_n=2, max_n=2)

In [46]:
# 특정 단어와 유사한 단어 추출 : 이정재
wv = model.wv
wv.most_similar('이정재')

[('정재영', 0.8506579399108887),
 ('이범수', 0.8397314548492432),
 ('정재', 0.8370071649551392),
 ('공유', 0.8365726470947266),
 ('정재형', 0.8220188617706299),
 ('송강호', 0.8196630477905273),
 ('박해일', 0.802772581577301),
 ('김범수', 0.7967075705528259),
 ('임성민', 0.792266309261322),
 ('김지민', 0.7780246138572693)]

In [47]:
# corpus에 없는 단어 확인 : 우주평화 
'우주평화' in wv.key_to_index

False

In [48]:
# corpus에 없는 단어의 임베딩 벡터 확인 
wv['우주평화']

array([ 0.1914269 ,  0.1860728 ,  0.12593931,  0.18663628, -0.16130134,
       -0.11598969, -0.1961977 ,  0.58747756,  0.36074576,  0.1287227 ,
       -0.09857217,  0.08718657,  0.00299251,  0.4163916 , -0.13992164,
       -0.2347496 , -0.02129206, -0.18993807, -0.0641426 ,  0.11315664,
        0.15295501, -0.06983434, -0.03780352, -0.22048816,  0.04815078,
       -0.15756145, -0.42226467, -0.30431196, -0.19651255, -0.27164346,
        0.16069016, -0.47310257,  0.31236234, -0.36893877, -0.11784068,
        0.03365862,  0.18324311,  0.2778246 , -0.40989056, -0.02491963,
       -0.08834364, -0.06328435, -0.1781125 ,  0.04878452, -0.16096899,
        0.07945447, -0.08593159,  0.13216922, -0.15957843,  0.00762285,
        0.47992048,  0.38991004, -0.23091011, -0.08668746, -0.08241071,
       -0.19514288,  0.35513696,  0.17578085,  0.1946751 ,  0.07655469,
       -0.05504971,  0.31071338, -0.40531287, -0.13801625, -0.25840122,
       -0.3768503 , -0.2206243 , -0.34182835, -0.00144067,  0.04

In [49]:
# corpus에 없는 단어와 유사한 단어추출 
wv.most_similar('우주평화', topn=20)

[('우주', 0.8189776539802551),
 ('우장', 0.807949423789978),
 ('우주비행사', 0.8074740171432495),
 ('평화', 0.8051568865776062),
 ('우방', 0.8048270344734192),
 ('우주인', 0.7847122550010681),
 ('투명인간', 0.784186065196991),
 ('우주여행', 0.7829931974411011),
 ('꽃밭', 0.7826895713806152),
 ('지구대', 0.7820442914962769),
 ('격동', 0.7788475751876831),
 ('지구인', 0.7772130966186523),
 ('산전수전', 0.7768232226371765),
 ('고물상', 0.774915337562561),
 ('지구촌', 0.7728344202041626),
 ('동병상련', 0.7720761895179749),
 ('낙동강', 0.7711057662963867),
 ('공직', 0.7709865570068359),
 ('포동포동', 0.7709451913833618),
 ('경제성장', 0.7701643705368042)]